In [ ]:
# Enable inline plotting for matplotlib
%matplotlib inline
# Import required libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
from glob import glob
import json
import matplotlib.pyplot as plt
import seaborn as sns

# Set base directory for Samsung watch data
samsung_base_dir = os.path.join('.', 'samsung_watch_data')
# Get list of all dump directories
samsung_dump_dirs = glob(os.path.join(samsung_base_dir, '*')) 
# Get name of first dump directory
samsung_dump_dir = os.path.basename(samsung_dump_dirs[0])
print(len(samsung_dump_dirs), 'dumps found, taking first:', samsung_dump_dir)

# Find all CSV files in the dump directory
samsung_csv_paths = glob(os.path.join(samsung_base_dir, samsung_dump_dir, '*.csv'))
print(len(samsung_csv_paths), 'csvs found')
# Find all JSON files in the jsons subdirectory
samsung_json_paths = glob(os.path.join(samsung_base_dir, samsung_dump_dir, 'jsons', '*', '*', '*.json'))
print(len(samsung_json_paths), 'jsons found')


Process CSV

In [ ]:
# Import display function from IPython for displaying dataframes
from IPython.display import display

# Define lambda function to read CSV files, skipping first row and not using index
sam_read_csv = lambda x: pd.read_csv(x, skiprows=1, index_col=False)

# Create dictionary of dataframes from CSV files, removing 'com.samsung.' prefix from filenames
all_csv_df = {os.path.basename(j).replace('com.samsung.', ''): sam_read_csv(j) for j in samsung_csv_paths}

# Loop through the dataframes dictionary and display samples
for k, v in all_csv_df.items():
    print(k, 'readings:', v.shape[0])  # Print filename and number of rows
    display(v.sample(2 if v.shape[0] > 2 else 1))  # Display 2 random rows (or 1 if data has only 1 row)


Step counts

In [ ]:
# Concatenate all dataframes containing 'step_daily_trend' in their keys
step_df = pd.concat([v for k, v in all_csv_df.items() if 'step_daily_trend' in k])

# Convert create_time and update_time columns to datetime format
for c_col in ['create_time', 'update_time']:
    step_df[c_col] = pd.to_datetime(step_df[c_col])

# Sort values by create_time in ascending order
step_df = step_df.sort_values('create_time', ascending=True)
pd.to_datetime(step_df['create_time'])
step_df.head(3)


In [ ]:
# Create a figure with one subplot, size 20x10 inches
fig, ax1 = plt.subplots(1, 1, figsize=(20, 10))
# Plot steps count over time
ax1.plot(step_df['create_time'], step_df['count'], '-', label='Steps')
# Plot distance in meters over time
ax1.plot(step_df['create_time'], step_df['distance'], '-', label='Meters')
# Plot calories burned over time
ax1.plot(step_df['create_time'], step_df['calorie'], '-', label='Calories')
# Add legend to show what each line represents
ax1.legend()
# Rotate x-axis date labels by 45 degrees for better readability
fig.autofmt_xdate(rotation=45)


In [ ]:
sns.pairplot(hue = 'deviceuuid', data = step_df[['count', 'distance', 'calorie', 'speed', 'deviceuuid']])

Day of week

In [ ]:
step_df['day_name'] = step_df['create_time'].dt.day_name()
step_df['is_weekend'] = step_df['day_name'].map(lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday')
sns.pairplot(hue = 'is_weekend', data = step_df[['count', 'distance', 'calorie', 'speed', 'day_name', 'is_weekend']])


In [ ]:
step_df['day_name'] = step_df['create_time'].dt.day_name()
step_df['is_weekend'] = step_df['day_name'].map(lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday')
sns.pairplot(hue = 'is_weekend', data = step_df[['count', 'distance', 'calorie', 'speed', 'day_name', 'is_weekend']])


In [ ]:
# Import necessary libraries for handling JSON files and iterables
from itertools import groupby, chain
import json

# Initialize empty dictionary to store JSON data
sam_json_dict = {}

# Group JSON files by folder name (removing 'com.samsung.' prefix)
for fold_id, files in groupby(samsung_json_paths,
                              lambda x: os.path.basename(os.path.dirname(x)[:-2]).replace('com.samsung.', '')):
    # Convert iterator to list
    c_files = list(files)

    # Load and parse JSON data from each file
    c_json_data = [json.load(open(c_file, 'r')) for c_file in c_files]
    # Chain lists together if data is list, otherwise keep as is
    sam_json_dict[fold_id] = list(chain(*c_json_data)) if isinstance(c_json_data[0], list) else c_json_data
    # Print summary info about the loaded data
    print(fold_id + ',', 'files:', len(c_files), 'readings:', len(sam_json_dict[fold_id]))
    print('\tPreview:', str(sam_json_dict[fold_id][0])[:80])


In [ ]:
step_df['hour'] = step_df['create_time'].dt.hour
sns.pairplot(hue = 'is_weekend', data = step_df[['count', 'distance', 'calorie', 'speed', 'hour', 'is_weekend']])

In [ ]:
stress_df = pd.DataFrame(sam_json_dict['shealth.stress'])
sns.pairplot(stress_df)